# Hyperliquid consolidated account analytics

Combines balances, account history, fills, funding, positions, and PnL across the configured Hyperliquid wallets. Combined account-value history is shown only over the dates available for every wallet; charts show the oldest wallet's account history before shared coverage begins, then the combined account value.

Daily rollups and calendar-period cutoffs use `America/New_York`. Previously saved UTC-aggregated daily totals are mapped to Eastern date labels when loaded; their original intraday split cannot be reconstructed.


## Wallets and implementation

Edit `WALLET_ADDRESSES` here. Both addresses found in the single-wallet notebook are included: the active address and the previously commented address.


In [76]:
from hyperliquid_multi_account_analytics_implementation import MultiWalletAnalytics

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

RISK_BUDGET_PCT = 5.0
ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
MARGIN_RESERVE_PCT = 20.0
AGGREGATE_RISK_PCT = 20.0

analytics=MultiWalletAnalytics(WALLET_ADDRESSES,data_dir='data')
analytics.position_risk_budget_pct=RISK_BUDGET_PCT
analytics.position_atr_period=ATR_PERIOD
analytics.position_atr_stop_multiple=ATR_STOP_MULTIPLE
analytics.margin_reserve_pct=MARGIN_RESERVE_PCT
analytics.aggregate_risk_pct=AGGREGATE_RISK_PCT


## Fetch and consolidate wallet data

Fetch each wallet once and prepare combined account history and fills.


In [77]:
%%capture
_ = analytics.fetch_data()


## Account / PnL

Combined account and risk summary. Daily PnL and calendar-period cutoffs use US Eastern time.


In [78]:
analytics.show_account_pnl()


=== ACCOUNT / PNL AND RISK ===


Metric,Value
Account value,"$2,918.78"
Portfolio value,"$3,952.53 (combined live wallet value)"
Account 1D (ET),2.39%
Realized today (ET),$13.74
Realized today (% of account value),0.47%
Unrealized PnL,$75.51
Unrealized PnL (% of account value),2.59%
Unrealized 1D change (ET),n/a
Average realized PnL / day,$-0.94
Average unrealized PnL change / day,n/a


## Account charts

Account value and PnL charts. Before all wallets have saved history, account value and weekly returns use the oldest wallet alone; from shared coverage onward they use the combined accounts.


In [79]:
analytics.show_account_charts()


Weekly return chart omits 2 return(s) below -100%;                     other data and stats are unchanged.


## Account vs BTC

The daily account-vs-BTC comparison uses a time-weighted return across all wallets, so deposits, withdrawals, and transfers do not move it.


In [80]:
analytics.show_account_vs_btc()


Lifetime account series is anchored to current account value; it includes fee-net realized trades and perp funding, but excludes historical cash flows and unrealized PnL.


## Traded volume vs realized PnL chart

Daily fill volume and realized PnL pooled across wallets.


In [81]:
analytics.show_volume_chart()


## Open positions

Show each wallet position separately, followed by fixed-fractional sizing grouped by asset and side and an aggregate summary. The summary includes cumulative open-position funding as a percentage of combined account equity and total position value. Reserve-adjusted available margin is combined margin collateral (USDC on unified accounts) less the configured reserve; the summary shows that capacity, margin used, remaining available margin in dollars and as a percentage of that capacity, and position value as a percentage of equity. Set `MARGIN_RESERVE_PCT` (20% by default) in the setup cell to change the reserve. The position table includes funding and fees since open as dollar amounts, plus fees since open as percentages of wallet equity and position value and position value as a percentage of wallet account value. Fee ratios are available for complete USDC-denominated fee history. Set `RISK_BUDGET_PCT` (5% of available margin per asset), `ATR_PERIOD` (14 completed daily candles), and `ATR_STOP_MULTIPLE` (1.25) in the setup cell. Stops are placed 1.25 ATR from the current mark, and take-profit is set the same distance in the favorable direction (1:1 risk/reward). Same-side positions across wallets are combined; long and short rows remain separate. If both sides are open for one asset, they share that asset's risk budget equally. `Open - Target` compares each row with its own budget only. `Addable Risk` and `Addable Value` also apply the aggregate risk cap (`AGGREGATE_RISK_PCT`, 20% of collateral by default); that room is one shared pool, so rows cannot all add their amounts.


In [82]:
analytics.show_open_positions()


Wallet,Coin,Dex,Side,Liq %,ROE,uPnL,Funding (since open),Funding (1h),Funding APR,Expected Price PnL (5d),Expected Funding PnL (5d),Expected PnL incl. funding (5d),Entry,Mark,Pos,Margin,Margin Type,Margin %,Fees,Fees / Equity (%),Fees / Position (%),Position / Account (%),Liq Px,Lev.,Sz
c248,IMX,,SHORT,69.31%,16.81%,$33.34,$0.68,0.001250%,10.95%,$12.10,$1.44,$13.53,0.12332,0.11918,$958,$192,Cross,28.55%,-$0.14,0.012%,0.015%,80.89%,0.20178,5.00x,"-8,040.50"
07f2,TRX,,SHORT,192.24%,-17.88%,-$12.46,$0.10,0.001250%,10.95%,-$1.46,-$1.33,-$2.79,0.313,0.318627,$709,$71,Cross,6.03%,-$0.10,0.004%,0.014%,25.62%,0.93116,10.00x,"-2,226.00"
07f2,TRUMP,,SHORT,192.75%,30.76%,$58.89,$0.06,0.001250%,10.95%,$11.06,$0.28,$11.34,1.75136,1.6174,$707,$177,Cross,15.04%,-$0.20,0.007%,0.028%,25.54%,4.7349,4.00x,-437.30
07f2,xyz:GOLD,xyz,LONG,n/a,9.11%,$8.21,-$0.20,0.001131%,9.91%,-$1.01,-$0.67,-$1.68,"3,909.8","3,960.7",$639,$91,Cross,7.77%,-$0.18,0.007%,0.028%,23.08%,0,7.00x,0.16
07f2,PENGU,,SHORT,213.55%,18.82%,$23.83,$0.33,0.001250%,10.95%,$2.04,$0.55,$2.60,0.007292,0.007019,$609,$122,Cross,10.37%,-$0.09,0.003%,0.015%,22.02%,0.022008,5.00x,"-86,832.00"
07f2,xyz:GOOGL,xyz,LONG,n/a,-52.97%,-$102.77,-$2.43,0.000625%,5.48%,$0.12,-$0.67,-$0.54,343.02,282.47,$479,$160,Cross,13.59%,-$0.05,0.002%,0.010%,17.32%,0,3.00x,1.70
07f2,MELANIA,,SHORT,291.18%,10.11%,$14.70,$0.10,0.001250%,10.95%,-$0.43,$0.24,-$0.19,0.07686,0.07427,$421,$140,Cross,11.95%,-$0.06,0.002%,0.015%,15.23%,0.29053,3.00x,"-5,674.80"
07f2,io:PRL,io,SHORT,14.43%,-2.76%,-$3.77,$6.29,0.004169%,36.52%,$18.60,$11.60,$30.20,1.50057,1.5144,$413,$138,Isolated,11.78%,-$0.04,0.001%,0.009%,14.94%,1.7329,2.99x,-273.00
07f2,xyz:JIO,xyz,LONG,31.45%,-1.05%,-$1.17,$5.93,-0.005783%,-50.66%,$25.88,$12.09,$37.97,14.54696,14.496,$332,$116,Isolated,9.85%,-$0.03,0.001%,0.009%,11.99%,9.9372,2.87x,22.90
c248,USELESS,,SHORT,204.51%,21.83%,$24.02,$0.29,0.001250%,10.95%,$12.37,$0.56,$12.93,0.132058,0.12247,$306,$102,Cross,15.20%,-$0.09,0.007%,0.028%,25.84%,0.37293,3.00x,"-2,500.00"


Funding and fees are cumulative since the position was opened. Fee ratios are shown only for complete USDC-denominated fees; otherwise they are n/a.
=== ACCOUNT NET EXPOSURE AND FUNDING ===


Net Direction,Long Positions,Short Positions,Long Notional ($),Short Notional ($),Gross Notional ($),Net Notional ($),Net Exposure / Gross (%),Net Funding Rate / 1h (%) (covered),"Net Funding APR (%) (simple, covered)",Estimated Net Funding PnL / hour ($),Estimated Net Funding PnL / day ($),Funding-Covered Notional ($),Funding Coverage (%)
NET SHORT,5,16,"$1,605.98","$5,568.52","$7,174.50","$-3,962.54",-55.23%,0.001136%,9.95%,$0.0815,$1.96,"$7,174.50",100.0%


Net direction compares long and short USD notionals; it is not delta- or beta-adjusted. Net funding rate is estimated net funding cash flow divided by funding-covered gross notional. Positive cash flow means estimated receipts. APR assumes current hourly rates persist; it is not a forecast.
=== FUNDING BY ASSET AND SIDE ===


Coin,Side,Positions,Size,Position Value ($),Current Funding / 1h (%),Funding APR (%),Estimated Funding PnL / hour ($),Estimated Funding PnL / day ($),Funding Coverage (%)
ADA,SHORT,1,"1,086",$204.86,-0.002600%,-22.77%,$-0.0053,$-0.13,100.0%
AERO,LONG,1,90,$54.70,-0.000167%,-1.46%,$0.0001,$0.00,100.0%
AVAX,SHORT,1,30,$251.74,0.001097%,9.61%,$0.0028,$0.07,100.0%
FOGO,SHORT,1,7.942e+04,$196.24,0.001250%,10.95%,$0.0025,$0.06,100.0%
GRASS,SHORT,1,452.3,$255.16,0.001250%,10.95%,$0.0032,$0.08,100.0%
IMX,SHORT,1,"8,040",$958.27,0.001250%,10.95%,$0.0120,$0.29,100.0%
INIT,SHORT,1,736,$103.86,0.001250%,10.95%,$0.0013,$0.03,100.0%
LIT,SHORT,1,9,$27.35,-0.000098%,-0.86%,$-0.0000,$-0.00,100.0%
MELANIA,SHORT,1,"5,675",$421.47,0.001250%,10.95%,$0.0053,$0.13,100.0%
PENGU,SHORT,1,8.683e+04,$609.39,0.001250%,10.95%,$0.0076,$0.18,100.0%


Funding rates are current hourly rates, weighted by position notional across wallets. Positive rates are paid by longs and received by shorts; estimated cash flows assume the current rate stays constant. APR is simple annualization, not a forecast.
Fixed-fractional sizing: risk budget 5.00% of combined account value per asset; target quantity uses 1.25 x daily ATR(14) plus the directionally adverse 95th-percentile five-day funding reserve. Opposing sides, when both open, split the asset budget equally.
Limited daily history; ATR uses all completed candles available (shorter than 14 days) for: xyz:JIO.
ATR stop and take-profit distances currently both use 1.25 x daily ATR from the current mark (1:1 price reward/risk before funding, fees, and slippage).
ATR stop-distance colors: green <2%, orange 2-5%, red >5%; red mark means the position is out of the money; stop prices are highlighted red. Open-risk includes the funding reserve but excludes fees, slippage, and gaps; the stop trigger it

Asset,Side,ATR (%),ATR Period Used,ATR Stop Distance (%),ATR Take Profit Distance (%),Mark to Take Profit (%),Funding Reserve / unit,Avg Entry,Mark,Stop,Take Profit,Open - Target (coins),Open - Target (%),Open Risk (price only) ($),Open Risk incl. funding ($)
ADA,SHORT,6.41%,14 days,8.01%,8.01%,8.01%,$0.000347462,0.193547,0.18865,0.20376,0.17354,"11,696.7",-91.50%,$16.41,$16.79
AERO,LONG,10.18%,14 days,12.72%,12.72%,12.72%,$0.00088612,0.832395,0.60814,0.53078,0.6855,"-2,435.64",-96.44%,$6.96,$7.04
AVAX,SHORT,6.53%,14 days,8.17%,8.17%,8.17%,$0.0029175,8.7597,8.3941,9.0797,7.7085,257.049,-89.55%,$20.57,$20.65
FOGO,SHORT,15.03%,14 days,18.79%,18.79%,18.79%,$0,0.002597,0.002471,0.0029353,0.0020067,"346,197",-81.34%,$36.88,$36.88
GRASS,SHORT,13.65%,14 days,17.06%,17.06%,17.06%,$0,0.60132,0.56413,0.66038,0.46788,"1,600.87",-77.97%,$43.54,$43.54
IMX,SHORT,9.01%,14 days,11.27%,11.27%,11.27%,$0,0.12332,0.11918,0.13261,0.10575,"6,675.23",-45.36%,$107.98,$107.98
INIT,SHORT,19.24%,14 days,24.05%,24.05%,24.05%,$0.000638034,0.18359,0.1413,0.17528,0.10732,"4,973.16",-87.11%,$25.01,$25.48
LIT,SHORT,11.96%,14 days,14.95%,14.95%,14.95%,$0.00344525,3.2512,3.0421,3.4967,2.5875,422.412,-97.91%,$4.09,$4.12
MELANIA,SHORT,7.74%,14 days,9.67%,9.67%,9.67%,$5.55458e-05,0.07686,0.07427,0.081451,0.067089,"21,633.5",-79.22%,$40.75,$41.07
PENGU,SHORT,8.37%,14 days,10.47%,10.47%,10.47%,$7.80807e-06,0.007292,0.007019,0.0077538,0.0062842,"179,296",-67.37%,$63.80,$64.48


Metric,Value
Wallets,2
Position value,"$7,174"
uPnL,$75.51
ROE,4.19%
Margin,"$1,802"
Margin % bal,46.47%
Available margin (80% of equity),"$3,162.03"
Available margin used (20% reserve),56.98%
Available margin remaining,"$1,360.42"
Available margin remaining (%),43.02%


ROE is unrealized PnL divided by used margin; a negative ROE means the open positions are currently at an unrealized loss relative to the margin supporting them.


## Net funding APR over time

Applies the open-positions `Net Funding APR (%) (simple, covered)` formula to Hyperliquid funding history: net funding cash flow divided by gross notional, annualized, pooled across wallets. The history records actual funding on positions held at the time; the summary table shows the current rate on current positions, so the two differ. Daily values use US Eastern days, and the 7-day line is notional-weighted. Positive means funding received.


In [ ]:
_ = analytics.show_funding_apr_chart()


Latest 7-day net funding APR: 29.05% (2026-11-02, America/New_York days). Lifetime: -7.61% on $-57.07 net funding.


## Realized PnL statistics

Pool spot/perp fills and signed funding across wallets.


In [83]:
analytics.show_realized_pnl()


=== REALIZED SPOT ===


Metric,Value
Realized PnL,$22.35 (0.57%)
24H,$0.00 (0.00%)
7D,$0.00 (0.00%)
30D,$22.35 (0.57%)
QTD,$22.35 (0.57%)
YTD,$22.35 (0.57%)
W,3
Win %,100.00%
Avg W,$7.45
L,0


=== REALIZED PERP ===


Metric,Value
Realized PnL,$-51.41 (-1.30%)
24H,$28.65 (0.72%)
7D,$367.08 (9.29%)
30D,$472.65 (11.96%)
QTD,$472.51 (11.95%)
YTD,$472.39 (11.95%)
W,191
Win %,67.49%
Avg W,$8.41
L,92



=== PERP PNL BREAKDOWN ===


Component,Amount ($),Contribution (% of |net PnL|)
Trading,$100.93,196.32%
Fees,$-107.23,-208.57%
Funding,$-45.11,-87.75%
Net perp PnL,$-51.41,


Liquidations: 0 of 283 closes (0.00%) | liquidation PnL: $0.00.


## Traded volume vs PnL

Combined turnover efficiency and correlations.


In [84]:
analytics.show_volume_summary()


=== TRADED VOLUME VS REALIZED PNL (ALL WALLETS) ===


Market,Volume ($),Realized PnL ($),Realized PnL / Volume (%),PnL / $1k volume ($),Active days,Daily Pearson r,Daily Spearman r
Perps,"$260,797.64",$-6.30,-0.00%,$-0.02,58,0.102,0.293
Spot,"$1,397.53",$22.35,1.60%,$16.00,5,-0.509,-0.462


## PnL attribution

Combined realized and open-position PnL by asset.


In [85]:
analytics.show_pnl_attribution()


=== CONSOLIDATED PNL ATTRIBUTION ===


Type,Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
Perp,BTC,12,$-475.21,-18.75%,$0.00,0.00%,$-475.21,-16.22%
Perp,kBONK,3,$-397.06,-15.67%,$0.00,0.00%,$-397.06,-13.56%
Perp,ZORA,21,$304.49,12.02%,$0.00,0.00%,$304.49,10.40%
Perp,KAITO,4,$-193.84,-7.65%,$0.00,0.00%,$-193.84,-6.62%
Perp,HYPE,11,$146.54,5.78%,$0.00,0.00%,$146.54,5.00%
Perp,VVV,40,$116.26,4.59%,$0.00,0.00%,$116.26,3.97%
Perp,ETH,15,$114.54,4.52%,$0.00,0.00%,$114.54,3.91%
Perp,xyz:GOOGL,0,$-2.76,-0.11%,$-102.77,-25.03%,$-105.53,-3.60%
Perp,io:ANTH,2,$-86.69,-3.42%,$0.00,0.00%,$-86.69,-2.96%
Perp,TRUMP,0,$0.07,0.00%,$61.95,15.09%,$62.02,2.12%


Net attributed PnL across wallets: $45.19.
